### **Research Mate (hereinafter RM)** is an AI paper summarizer agent performing the following tasks:
1. **Customized Paper Search**: Simply input your keywords of interest, and RM will scour through papers published in the last seven days, retrieving those containing at least two of your specified keywords. 

2. **Paper Prioritization**: RM doesn't dump a list of papers on you. It analyzes the title and abstract of each paper, evaluates its relevance to your keywords, and presents you with the top 10 most relevant papers.(Prioritization method: Papers are ranked by the average cosine similarity score between their titles/abstracts and the input keywords.) 

3. **Summary and Original Link**: RM provides a concise summary of each selected paper, along with a direct link to the original article.  Quickly grasp the paper's essence from the summary, and dive into the full details by clicking the link.

<br>

#### <span style="color: green;">**&#10004; Generative AI capabilities in RM** </span>
Here's a brief overview of the generative AI capabilities used in this paper summarizer code:

* **Structured output/JSON mode/controlled generation**: The code processes search results from the arXiv API, which are in JSON format.
* **Few-shot prompting**: Keyword-based search acts as a form of prompting to the arXiv API.
* **Document understanding**: The summarization pipeline from the Transformers library extracts key information from paper abstracts.
* **Function Calling**: The code calls external functions like the arXiv API and the summarization pipeline.
* **Agents**: The code automates the paper search, evaluation, summarization, and output process.
* **Context caching**: Embedding and summarization models are loaded once and reused.
* **Gen AI evaluation**: Paper importance is evaluated using cosine similarity.
* **Retrieval augmented generation (RAG) & Grounding**: The code retrieves information from arXiv to generate summaries, grounding the output in external sources.
* **Embeddings**: Sentence-transformers are used to convert text to numerical vectors for evaluating paper importance.
* **Vector search/vector store/vector database**: Cosine similarity calculation is a form of vector search to measure the relevance between papers and keywords.

In [ ]:
# Install necessary libraries (for environments where they are not pre-installed)
!pip install arxiv scholarly hf_xet

#### The section below imports the required Python libraries for the script
* `torch` : For GPU acceleration
* `datetime`: For handling dates
* `arxiv`: For searching arXiv papers
* `scholarly`: For searching Google Scholar papers
* `sentence_transformers`: For generating sentence embeddings
* `transformers`: For text summarization
* `IPython.display`: For displaying HTML output
* `itertools`: for combinations

In [ ]:
# Import PyTorch library
import torch

# Check GPU availability and set device
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"PyTorch is using GPU: {torch.cuda.get_device_name(0)}")
    print(f"PyTorch CUDA version: {torch.version.cuda}")
else:
    device = torch.device("cpu")
    print("PyTorch is using CPU")

# Import other necessary libraries
import os
import json
from datetime import datetime, timedelta
import arxiv  # arXiv paper search
#from scholarly import scholarly  # Google Scholar paper search (not used)
from sentence_transformers import SentenceTransformer  # Sentence embedding generation
from transformers import pipeline  # Text summarization
from IPython.display import HTML, display  # For displaying HTML in the notebook
from itertools import combinations  # Keyword combination generation

#### Configuration
This section defines the constants and parameters for the paper search and summarization process.

In [ ]:
# --- Search Configuration ---
NUM_ARTICLES_PER_KEYWORD = 50   # Number of articles to retrieve per keyword (for flexible search)
NUM_ARTICLES_TO_SUMMARIZE = 10  # Number of top papers to summarize
MIN_KEYWORDS_MATCH = 2        # Minimum number of keywords that must be present in the paper for it to be considered
SEARCH_TERM = 7               # Number of days to look back for papers

# --- Date Configuration ---
TODAY = datetime.now() # Today's date
LAST_WEEK = TODAY - timedelta(days=SEARCH_TERM) # Search start date (SEARCH_TERM days ago)
DATE_QUERY = f"submittedDate:[{LAST_WEEK.strftime('%Y%m%d')} TO {TODAY.strftime('%Y%m%d')}]" # Date query for arXiv search API (related to structured output/JSON mode, controlled generation)
SEARCH_START_DATE_STR = LAST_WEEK.strftime('%Y-%m-%d') # Human-readable date strings
SEARCH_END_DATE_STR = TODAY.strftime('%Y-%m-%d')

#### Load Models
This section loads the pre-trained models for sentence embedding and text summarization.

In [ ]:
# --- Model Loading ---
# Load the sentence embedding model
embedding_model_name = 'all-mpnet-base-v2'
embedding_model = SentenceTransformer(embedding_model_name, device=device)

# Load the summarization model
summarization_model_name = 'facebook/bart-large-cnn'
summarizer = pipeline("summarization", model=summarization_model_name, device=device)

#### Define Functions
This section defines the functions for searching papers, evaluating importance, and summarizing.

##### **arXiv Paper Search Function**
Searches for papers on arXiv based on the given keywords and date range. It requires a minimum number of keywords to be present in the results.

In [ ]:
# --- arXiv Paper Search Function (Find papers with at least two keywords in AND relation) ---
def search_arxiv_papers_at_least_two_and(keywords, date_query, max_results):
    all_results = []
    if len(keywords) >= MIN_KEYWORDS_MATCH:
        client = arxiv.Client()  # Create an arXiv client
        for i in range(MIN_KEYWORDS_MATCH, len(keywords) + 1):
            for keyword_combination in combinations(keywords, i):
                combined_query = " AND ".join(f"({keyword})" for keyword in keyword_combination)
                query = f"{combined_query} AND {date_query}"
                print(f"Searching with query (AND of {i} keywords): {query}")
                search = arxiv.Search(
                    query=query,
                    max_results=max_results // (len(keywords) - MIN_KEYWORDS_MATCH + 1) if (len(keywords) - MIN_KEYWORDS_MATCH + 1) > 0 else max_results
                )
                for result in client.results(search):  # Use Client.results()
                    if result not in all_results:
                        all_results.append(result)
    elif keywords:
        # If less than two keywords are provided, inform the user
        print("Please enter at least two keywords for AND search.")
    return all_results

##### **Paper Importance Evaluation Function**
Evaluates the importance of each paper based on the cosine similarity between the paper's title and abstract and the given keywords.

In [ ]:
# --- Paper Importance Evaluation Function (using sentence embeddings) ---
def evaluate_paper_importance(papers, keywords):
    from sklearn.metrics.pairwise import cosine_similarity
    import numpy as np

    keyword_embeddings = embedding_model.encode(keywords)

    paper_scores = []
    for paper in papers:
        title_abstract = f"{paper.title}. {paper.summary}"
        paper_embedding = embedding_model.encode([title_abstract])[0]
        similarity_scores = [cosine_similarity(paper_embedding.reshape(1, -1), key_emb.reshape(1, -1))[0][0] for key_emb in keyword_embeddings]
        importance_score = np.mean(similarity_scores)
        paper_scores.append((paper, importance_score))

    # Sort papers by importance score in descending order
    sorted_papers = sorted(paper_scores, key=lambda item: item[1], reverse=True)
    return [paper for paper, score in sorted_papers]

##### **Paper Summarization Function**
Summarizes the paper's abstract using the pre-trained summarization model.

In [ ]:
# --- Paper Summarization Function (using transformers pipeline) ---
def summarize_paper(paper):
    try:
        summary = summarizer(paper.summary, max_length=150, min_length=30, do_sample=False)[0]['summary_text'] # Generate summary
        return summary
    except Exception as e:
        print(f"Summarization failed: {e}")
        return "Summarization failed"

#### Main Function
This is the main function that orchestrates the paper search, evaluation, and summarization process.

In [ ]:
def main():
    print(f"Today's Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S %Z%z')}")  # Print current date

    if __name__ == "__main__":
        # Get user input for keywords
        print(f"Enter keywords separated by commas (searching papers from the last {SEARCH_TERM} days, with at least {MIN_KEYWORDS_MATCH} in AND):")
        keywords_input = input()
        USER_KEYWORDS = [keyword.strip() for keyword in keywords_input.split(',')]  # Convert input string to keyword list
        print(f"Set keywords: {USER_KEYWORDS}")
    else:
        USER_KEYWORDS = ["example", "paper"]  # Assign default values

    # 1. Search for Latest Papers (at least two keywords in AND relation)
    print(f"Searching for latest papers (at least {MIN_KEYWORDS_MATCH} keywords in AND) from {SEARCH_START_DATE_STR} to {SEARCH_END_DATE_STR}...")
    all_papers = search_arxiv_papers_at_least_two_and(USER_KEYWORDS, DATE_QUERY, NUM_ARTICLES_PER_KEYWORD)  # Search for papers
    print(f"Total {len(all_papers)} papers found with at least {MIN_KEYWORDS_MATCH} keywords in AND relation.")

    if not all_papers:
        print("No papers found matching the criteria. Please enter different keywords.")
        #return  # Exit function if no papers found
        continue # If no papers are found, ask for keywords again
    else:
        break # If papers are found, exit the loop 

    # 2. Evaluate Paper Importance
    print("Evaluating paper importance...")
    top_papers = evaluate_paper_importance(all_papers, USER_KEYWORDS)  # Evaluate paper importance
    top_n_papers = top_papers[:NUM_ARTICLES_TO_SUMMARIZE]  # Select top N papers
    print(f"{len(top_n_papers)} most important papers selected.")

    # 3. Summarize Papers
    summarized_papers = []
    for paper in top_n_papers:
        summary = summarize_paper(paper)  # Summarize paper
        summarized_papers.append({"title": paper.title, "summary": summary, "url": paper.pdf_url})  # Store summary results

    # 4. Output Results (in HTML format)
    if summarized_papers:
        print("\n--- Today's Personalized Academic Paper Summaries ---")
        html_output = f"""
        <html>
          <head></head>
          <body>
            <h2>Today's Personalized Academic Paper Summaries</h2>
            <p>Search period: {SEARCH_START_DATE_STR} to {SEARCH_END_DATE_STR}</p>
        """
        for paper in summarized_papers:
            html_output += f"""
            <h3>{paper['title']}</h3>
            <p>{paper['summary']}</p>
            <p><a href="{paper['url']}">Original Link</a></p>
            <hr>
            """
        html_output += """
          </body>
        </html>
        """
        display(HTML(html_output))  # Display HTML output
    else:
        print("No papers to summarize.")

# --- Main Execution ---
if __name__ == "__main__":
    main()  # Call main function